# VeriClue — chuẩn bị lớp giọng thật NES2Net

Chạy trên Colab/Kaggle CPU, không cần GPU, Drive mount hoặc token.
Chỉ tải dữ liệu publisher DNU đã ghim revision; giữ split theo người, kiểm SHA/
PCM và xuất bundle. Chưa có lớp tấn công, không fit/chọn ngưỡng hoặc sinh clone.
Giữ notebook private, không chia sẻ audio/output công khai.
Chỉ tải bundle sau khi cell chạy xong và kiểm hash; test vẫn dành đánh giá cuối.


In [ ]:
# Generated from scripts/prepare_nes2net_data.py; do not edit embedded code.
import hashlib, json, os, runpy
from pathlib import Path
SOURCE_COMMIT = '140beb55d27e7c4ae2f4bf8883117c7c2a1553e6'
SOURCE_DIRTY = False
SOURCE_SHA256 = '52c7a43c04f67893fb109f693a77e27805d07db2e1fbee1465dde7525a1baac5'
if SOURCE_DIRTY:
    raise ValueError("Commit preparation source and regenerate notebook before collecting data")
source = '"""Prepare a pinned DNU bona-fide bundle on Colab/Kaggle; never fits a model.\n\nKeep the publisher\'s original speaker partitions, choose by a fixed ID hash,\nand audit unchanged native WAVs. Public release does not grant a new consent\nscope for cloning or authenticating the participants. No clone is generated.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport re\nimport urllib.request\nimport uuid\nimport wave\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom collections.abc import Callable\nfrom concurrent.futures import ThreadPoolExecutor\nfrom datetime import UTC, datetime\nfrom pathlib import Path, PurePosixPath, PureWindowsPath\nfrom urllib.parse import quote\n\nREPOSITORY = "TrangLe2108/DNU-ViSpeech"\nREVISION = "33b2a469901cb71b243403254d3cfef25170526c"\nBASE = f"https://huggingface.co/datasets/{REPOSITORY}/resolve/{REVISION}/"\nVERSION = "dnu-bonafide-native16k-readonly-v1"\nSPLITS = {"train": "calib_fit", "validation": "calib_ltt", "test": "test"}\nMAX_FETCH_BYTES = 64 * 1024 * 1024\n\n\ndef _relative(name: str) -> str:\n    path = PurePosixPath(name)\n    if (\n        not name\n        or path.is_absolute()\n        or PureWindowsPath(name).drive\n        or ".." in path.parts\n        or "\\\\" in name\n        or "\\x00" in name\n    ):\n        raise ValueError("Unsafe relative data path")\n    return path.as_posix()\n\n\ndef select_samples(metadata: dict[str, bytes], *, per_speaker: int, seed: int) -> list[dict]:\n    """Choose before inference; reject malformed provenance or overlapping people."""\n    if type(per_speaker) is not int or not 1 <= per_speaker <= 1000 or type(seed) is not int:\n        raise ValueError("Explicit bounded sample count and integer seed required")\n    if set(metadata) != set(SPLITS):\n        raise ValueError("All publisher metadata splits are required")\n    memberships, seen_ids, selected = {}, set(), []\n    for original, target in SPLITS.items():\n        groups = defaultdict(list)\n        reader = csv.DictReader(io.StringIO(metadata[original].decode("utf-8-sig")))\n        if not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames):\n            raise ValueError("Missing or duplicate metadata columns")\n        for row in reader:\n            try:\n                name = _relative(row["file_name"])\n                subject, sample = row["speaker_id"], row["audio_id"]\n                digest = row["checksum_sha256"]\n                if (\n                    re.fullmatch(r"speaker[0-9]{3}", subject) is None\n                    or re.fullmatch(r"[A-Za-z0-9._-]{1,120}", sample) is None\n                    or not sample.startswith(subject + "_")\n                    or re.fullmatch(r"[0-9a-f]{64}", digest) is None\n                    or row["split"]\n                    not in ({"validation", "dev"} if original == "validation" else {original})\n                ):\n                    raise ValueError("Invalid publisher provenance")\n                if sample in seen_ids or memberships.setdefault(subject, original) != original:\n                    raise ValueError("Duplicate sample or subject across publisher splits")\n                seen_ids.add(sample)\n                eligible = (\n                    row["review_status"] == "accepted"\n                    and row["qc_status"] == "passed"\n                    and int(row["sample_rate"]) == 16000\n                    and int(row["num_channels"]) == 1\n                    and 1 <= float(row["duration"]) <= 30\n                )\n                if eligible:\n                    groups[subject].append((sample, name, digest))\n            except (KeyError, TypeError) as exc:\n                raise ValueError("Incomplete publisher metadata") from exc\n        if not groups:\n            raise ValueError("Publisher split has no eligible recordings")\n        for subject, candidates in sorted(groups.items()):\n            candidates.sort(\n                key=lambda item: hashlib.sha256(f"{seed}:{item[0]}".encode()).hexdigest()\n            )\n            for sample, name, digest in candidates[:per_speaker]:\n                selected.append(\n                    {\n                        "sample_id": "DNU:" + sample,\n                        "subject_ids": ["DNU:" + subject],\n                        "source_group": "DNU:" + sample,\n                        "split": target,\n                        "label": 0,\n                        "attack_type": "bonafide",\n                        "audio": f"DNU-ViSpeech/data/{original}/{name}",\n                        "audio_sha256": digest,\n                        "source_dataset": REPOSITORY,\n                        "source_revision": REVISION,\n                        "rights_reference": f"https://huggingface.co/datasets/{REPOSITORY}/blob/{REVISION}/LICENSE; research use, no cloning authorization asserted",\n                        "usage": "test_only" if target == "test" else "calibration_evaluation",\n                    }\n                )\n    return selected\n\n\ndef fetch_bytes(url: str) -> bytes:\n    """Bound HTTPS transfers from the pinned publisher; no account token required."""\n    if not url.startswith(BASE):\n        raise ValueError("Only the pinned publisher revision is allowed")\n    request = urllib.request.Request(url, headers={"User-Agent": "VeriClue-data-preparation/1"})\n    with urllib.request.urlopen(request, timeout=45) as response:\n        raw = response.read(MAX_FETCH_BYTES + 1)\n    if len(raw) > MAX_FETCH_BYTES:\n        raise ValueError("Publisher file exceeds download budget")\n    return raw\n\n\ndef _write_once(path: Path, raw: bytes, *, root: Path) -> None:\n    if not path.resolve().is_relative_to(root.resolve()):\n        raise ValueError("Evidence output escapes bundle storage")\n    if path.exists():\n        if not path.is_file() or path.read_bytes() != raw:\n            raise ValueError("Existing evidence differs; never overwrite it")\n        return\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("xb") as stream:\n        stream.write(raw)\n\n\ndef _wav_identity(raw: bytes) -> dict:\n    if len(raw) > 2 * 1024 * 1024:\n        raise ValueError("Native WAV exceeds recording limit")\n    try:\n        with wave.open(io.BytesIO(raw), "rb") as reader:\n            count = reader.getnframes()\n            if (\n                reader.getnchannels() != 1\n                or reader.getsampwidth() != 2\n                or reader.getframerate() != 16000\n                or reader.getcomptype() != "NONE"\n                or not 16000 <= count <= 480000\n            ):\n                raise ValueError("Expected unchanged mono16k PCM16,1..30seconds")\n            pcm = reader.readframes(count)\n            if len(pcm) != count * 2:\n                raise ValueError("Truncated PCM")\n    except (wave.Error, EOFError) as exc:\n        raise ValueError("Invalid native WAV") from exc\n    return {"input_samples": count, "pcm_s16le_sha256": hashlib.sha256(pcm).hexdigest()}\n\n\ndef prepare(\n    root: Path,\n    *,\n    per_speaker: int,\n    seed: int,\n    commit: str,\n    fetch: Callable[[str], bytes] = fetch_bytes,\n    workers: int = 4,\n) -> dict:\n    """Prepare bona-fide data only; preserve failed/altered files and expose missing attacks."""\n    if (\n        re.fullmatch(r"[0-9a-f]{40}", commit) is None\n        or type(workers) is not int\n        or not 1 <= workers <= 8\n    ):\n        raise ValueError("Clean source commit and bounded workers required")\n    root = root.resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    evidence = {}\n    metadata = {}\n    for name in (\n        "LICENSE",\n        "PUBLIC_RELEASE_APPROVAL.md",\n        *(f"data/{split}/metadata.csv" for split in SPLITS),\n    ):\n        raw = fetch(BASE + quote(name, safe="/"))\n        _write_once(root / "source" / name, raw, root=root)\n        evidence[name] = hashlib.sha256(raw).hexdigest()\n        if name.endswith("metadata.csv"):\n            metadata[name.split("/")[1]] = raw\n    if b"CC BY-NC 4.0" not in (root / "source/LICENSE").read_bytes():\n        raise ValueError("Publisher license requires review")\n    samples = select_samples(metadata, per_speaker=per_speaker, seed=seed)\n    manifest = {"schema_version": 1, "dataset": "DNU-ViSpeech-bonafide-only-v1", "samples": samples}\n    encoded = (json.dumps(manifest, ensure_ascii=False, indent=2) + "\\n").encode()\n    _write_once(root / "manifest.json", encoded, root=root)\n\n    def download(sample):\n        base = {"sample_id": sample["sample_id"], "audio_sha256": sample["audio_sha256"]}\n        try:\n            target = (root / _relative(sample["audio"])).resolve()\n            if not target.is_relative_to(root):\n                raise ValueError("Audio output escapes bundle")\n            reused = target.exists()\n            raw = (\n                target.read_bytes()\n                if reused\n                else fetch(BASE + quote(sample["audio"].removeprefix("DNU-ViSpeech/"), safe="/"))\n            )\n            if hashlib.sha256(raw).hexdigest() != sample["audio_sha256"]:\n                raise ValueError("Changed publisher bytes")\n            identity = _wav_identity(raw)\n            if not reused:\n                _write_once(target, raw, root=root)\n            return {**base, **identity, "status": "verified", "reused": reused}\n        except Exception as exc:\n            return {**base, "status": "error", "error_type": type(exc).__name__}\n\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        rows = list(pool.map(download, samples))\n    errors = [row for row in rows if row["status"] == "error"]\n    seen_pcm, seen_files = set(), set()\n    for row in rows:\n        if row["status"] != "verified":\n            continue\n        pcm, digest = row["pcm_s16le_sha256"], row["audio_sha256"]\n        if pcm in seen_pcm or digest in seen_files:\n            errors.append({"sample_id": row["sample_id"], "error_type": "DuplicatePCMOrFile"})\n        seen_pcm.add(pcm)\n        seen_files.add(digest)\n    report = {\n        "preparation_version": VERSION,\n        "source_commit": commit,\n        "script_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n        "publisher": REPOSITORY,\n        "publisher_revision": REVISION,\n        "source_hashes": evidence,\n        "manifest_sha256": hashlib.sha256(encoded).hexdigest(),\n        "seed": seed,\n        "per_speaker": per_speaker,\n        "sample_count": len(samples),\n        "split_counts": dict(Counter(s["split"] for s in samples)),\n        "speaker_counts": {\n            split: len({s["subject_ids"][0] for s in samples if s["split"] == split})\n            for split in SPLITS.values()\n        },\n        "downloaded_audio_files": sum(r["status"] == "verified" and not r["reused"] for r in rows),\n        "reused_audio_files": sum(r["status"] == "verified" and r["reused"] for r in rows),\n        "pcm_duplicate_check": "failed_or_incomplete" if errors else "passed",\n        "state": "incomplete" if errors else "bonafide_only",\n        "calibration_ready": False,\n        "missing": ["attack_data"],\n        "clone_scope_confirmed": False,\n        "errors": errors,\n        "rows": rows,\n    }\n    reports = root / "runs"\n    reports.mkdir(exist_ok=True)\n    name = datetime.now(UTC).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:8] + ".json"\n    _write_once(\n        reports / name,\n        (json.dumps(report, ensure_ascii=False, indent=2) + "\\n").encode(),\n        root=root,\n    )\n    return report\n\n\ndef archive_bundle(root: Path, output: Path) -> str:\n    """Export a new bundle without replacing existing archives or including itself."""\n    root, output = root.resolve(), output.resolve()\n    if output.is_relative_to(root):\n        raise ValueError("Archive must be outside bundle storage")\n    reports = list((root / "runs").glob("*.json"))\n    latest = max(reports, key=lambda p: p.stat().st_mtime_ns) if reports else None\n    report = json.loads(latest.read_text("utf-8")) if latest else {}\n    if report.get("state") != "bonafide_only" or report.get("errors"):\n        raise ValueError("Cannot archive incomplete preparation")\n    manifest_raw = (root / "manifest.json").read_bytes()\n    if hashlib.sha256(manifest_raw).hexdigest() != report["manifest_sha256"]:\n        raise ValueError("Manifest changed after preparation")\n    manifest = json.loads(manifest_raw)\n    expected = {\n        "manifest.json": report["manifest_sha256"],\n        latest.relative_to(root).as_posix(): hashlib.sha256(latest.read_bytes()).hexdigest(),\n    }\n    expected |= {\n        "source/" + _relative(name): digest for name, digest in report["source_hashes"].items()\n    }\n    expected |= {\n        _relative(sample["audio"]): sample["audio_sha256"] for sample in manifest["samples"]\n    }\n    for name, digest in expected.items():\n        path = (root / name).resolve()\n        if not path.is_relative_to(root) or hashlib.sha256(path.read_bytes()).hexdigest() != digest:\n            raise ValueError("Bundle evidence changed before archive")\n    with zipfile.ZipFile(output, "x", compression=zipfile.ZIP_DEFLATED) as target:\n        for name, digest in sorted(expected.items()):\n            raw = (root / name).read_bytes()\n            if hashlib.sha256(raw).hexdigest() != digest:\n                raise ValueError("Bundle evidence changed during archive")\n            target.writestr(name, raw)\n    with output.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output-root", required=True, type=Path)\n    parser.add_argument("--source-commit", required=True)\n    parser.add_argument("--per-speaker", type=int, default=20)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--workers", type=int, default=4)\n    parser.add_argument("--archive", type=Path)\n    args = parser.parse_args(argv)\n    report = prepare(\n        args.output_root,\n        per_speaker=args.per_speaker,\n        seed=args.seed,\n        commit=args.source_commit,\n        workers=args.workers,\n    )\n    if args.archive and not report["errors"]:\n        report["archive_sha256"] = archive_bundle(args.output_root, args.archive)\n    print(\n        json.dumps(\n            {key: value for key, value in report.items() if key != "rows"}, ensure_ascii=False\n        )\n    )\n    return 1 if report["errors"] else 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
if hashlib.sha256(source.encode()).hexdigest() != SOURCE_SHA256:
    raise ValueError("Embedded source changed")
output_root = Path(os.environ.get("VERICLUE_DATA_OUTPUT", "/content/vericlue-dnu-bonafide"))
output_root.parent.mkdir(parents=True, exist_ok=True)
script = output_root.parent / ("prepare_nes2net_data-" + SOURCE_SHA256[:12] + ".py")
if script.exists() and script.read_text("utf-8") != source:
    raise ValueError("Existing embedded source differs")
if not script.exists():
    with script.open("x", encoding="utf-8", newline="\n") as handle:
        handle.write(source)
job = runpy.run_path(str(script))
report = job["prepare"](output_root, per_speaker=20, seed=42, commit=SOURCE_COMMIT, workers=4)
print(json.dumps({key: value for key, value in report.items() if key != "rows"}, ensure_ascii=False))
if report["errors"]:
    raise ValueError("Preparation incomplete; keep evidence and rerun to resume, do not fit")
archive = output_root.parent / (output_root.name + "-" + __import__("uuid").uuid4().hex[:8] + ".zip")
archive_sha256 = job["archive_bundle"](output_root, archive)
print(json.dumps({"archive": str(archive), "sha256": archive_sha256, "calibration_ready": False, "missing": ["attack_data"]}))


In [ ]:
# Chỉ tải bundle đã kiểm; không cài artifact hay thay cấu hình API.
from google.colab import files
files.download(str(archive))
